In [2]:
import numpy as np
# import yfinance as yf
import pandas as pd

In [3]:
# Ticker = "NQ=F"

# df = yf.download(Ticker, period="max", interval="15m")

# df.columns.names = [None, None]

# df.columns = df.columns.get_level_values(0)

# df = df.drop(columns=["Volume"])

In [4]:
df = pd.read_csv("data.csv")

df.set_index("timestamp ET", inplace=True)

df.index.name = "Datetime"

df = df[["Open", "High", "Low", "Close"]]

df.index = pd.to_datetime(
    df.index,
    format="%m/%d/%Y %H:%M"
).tz_localize("America/New_York")

In [5]:
# df["Side"] = "None"

# df = df.tail(500) # for testing purposes

df

,Open,High,Low,Close
Datetime,,,,
2022-12-26 18:01:00-05:00,13759.00,13794.75,13759.00,13788.50
2022-12-26 18:02:00-05:00,13790.25,13794.00,13784.00,13790.25
2022-12-26 18:03:00-05:00,13789.75,13791.75,13782.50,13784.25
2022-12-26 18:04:00-05:00,13783.00,13783.25,13776.00,13777.25
2022-12-26 18:05:00-05:00,13777.75,13783.50,13777.50,13783.50
...,...,...,...,...
2025-12-11 20:48:00-05:00,25922.75,25926.50,25921.50,25924.75
2025-12-11 20:49:00-05:00,25924.00,25925.25,25920.75,25923.75
2025-12-11 20:50:00-05:00,25924.00,25924.25,25920.75,25922.75


In [6]:
RRR = 2
RANGE_LENGTH = 10

### Strategy

In [7]:
high = df["High"].to_numpy()
low = df["Low"].to_numpy()

outcome = np.full(len(df), 0.0)

MAX_LOOKAHEAD = 3*60

for root in range(RANGE_LENGTH - 1, len(df)):
    range_high = high[root - RANGE_LENGTH + 1:root + 1].max()
    range_low = low[root - RANGE_LENGTH + 1:root + 1].min()

    risk = range_high - range_low

    is_long = False
    is_short = False

    counter = 0

    for i in range(root + 1, len(df)):
        counter += 1

        if counter >= MAX_LOOKAHEAD: break

        if not is_long and not is_short:

            if high[i] >= range_high and low[i] > range_low: is_long = True

            elif low[i] <= range_low and high[i] < range_high: is_short = True

            elif low[i] > range_low and high[i] < range_high: continue

            else: break  # Trade triggered and SL hit right after it

        if is_long:
            if low[i] <= range_low: break # SL hit

            if high[i] - range_high > RRR * risk: # TP hit
                outcome[root] = RRR
                break

        elif is_short:
            if high[i] >= range_high: break # SL hit

            if range_low - low[i] > RRR * risk: # TP hit
                outcome[root] = RRR
                break

In [8]:
df["Outcome"] = outcome

result_df = (
    df.groupby(df.index.strftime("%H:%M"))["Outcome"]
      .apply(lambda x: (x == RRR).mean())
      .rename("Win Rate")
      .to_frame()
)

result_df.index.name = "Time"

result_df["Expectancy"] = (RRR + 1)* result_df["Win Rate"] - 1
result_df["Win Rate"] = result_df["Win Rate"] * 100
result_df = result_df.sort_values("Win Rate", ascending=False)

result_df.head(20)

,Win Rate,Expectancy
Time,,
17:00,45.776567,0.373297
16:59,41.904762,0.257143
16:58,39.047619,0.171429
16:57,38.367347,0.151020
16:56,38.095238,0.142857
07:01,37.124183,0.113725
01:21,36.993464,0.109804
01:22,36.910995,0.107330
09:25,36.470588,0.094118
